# 사용자 지정 RAG 워크플로 (AWS Bedrock Knowledge Base)

사용자 질문을 AWS Bedrock Knowledge Base에서 검색해 컨텍스트로 가져오고,
그 컨텍스트와 질문을 합쳐 프롬프트를 증강한 뒤 LLM으로 답변을 생성한다.

**흐름:** 사용자 입력 → Retrieve API → 컨텍스트 → 프롬프트 증강 → LLM → 답변

- 검색(Retrieve): `bedrock-agent-runtime` 의 `retrieve` API — 쿼리 임베딩 생성과 유사 문서 검색을 Bedrock이 관리형으로 처리
- 생성(Generate): `bedrock-runtime` 의 `converse` API — 증강 프롬프트로 LLM 답변 생성

각 셀을 위에서부터 순서대로 실행하면서 중간 결과(검색 문서, 컨텍스트, 프롬프트, 답변)를 확인하세요.

> 사전 조건: Bedrock Knowledge Base가 이미 생성/동기화되어 있고, 실행 환경에 Bedrock 접근 권한과 사용 모델 액세스가 활성화되어 있어야 합니다.

## 셀 1 — 패키지 임포트

`boto3`가 설치되어 있지 않다면 아래 설치 줄의 주석을 해제해 먼저 실행하세요.

In [ ]:
# %pip install boto3
import boto3

print("boto3 version:", boto3.__version__)

## 셀 2 — 설정

리전, Knowledge Base ID, LLM 모델 ID, 검색 개수를 설정합니다.

> 자격 증명(Access Key 등)은 코드에 하드코딩하지 않습니다. 표준 AWS 자격 증명 체인(환경 변수, `~/.aws/credentials`, 프로파일, IAM 역할)을 사용하세요.

In [ ]:
# TODO: 아래 값을 본인 환경에 맞게 채워 넣으세요.
AWS_REGION = "us-east-1"
KNOWLEDGE_BASE_ID = "YOUR_KNOWLEDGE_BASE_ID"  # 예: "ABCDEFGHIJ"
MODEL_ID = "anthropic.claude-3-5-sonnet-20240620-v1:0"  # 모델 ID 또는 inference profile ID
TOP_K = 3  # 컨텍스트로 사용할 상위 문서 개수

print("region:", AWS_REGION)
print("knowledge base:", KNOWLEDGE_BASE_ID)
print("model:", MODEL_ID)
print("top_k:", TOP_K)

## 셀 3 — 클라이언트 초기화

검색용(`bedrock-agent-runtime`)과 생성용(`bedrock-runtime`) 클라이언트를 만듭니다.

In [ ]:
agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

print("clients ready")

## 셀 4 — 사용자 질문

검색하고 답변받을 질문을 입력합니다.

In [ ]:
question = "여기에 질문을 입력하세요"
print(question)

## 셀 5 — Retrieve (검색)

Knowledge Base의 `retrieve` API를 호출합니다. 쿼리 임베딩 생성과 유사 문서 검색은 Bedrock이 관리형으로 수행합니다.

In [ ]:
def retrieve(query, kb_id, top_k=TOP_K):
    resp = agent_runtime.retrieve(
        knowledgeBaseId=kb_id,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {"numberOfResults": top_k}
        },
    )
    return resp["retrievalResults"]


results = retrieve(question, KNOWLEDGE_BASE_ID)

if not results:
    print("관련 문서를 찾지 못했습니다.")
else:
    for i, r in enumerate(results, 1):
        score = r.get("score")
        location = r.get("location")
        print(f"[문서 {i}] score={score} location={location}")
        print(r["content"]["text"][:300])
        print("-" * 40)

## 셀 6 — 컨텍스트 구성

검색된 문서 청크들을 하나의 컨텍스트 문자열로 합칩니다.

In [ ]:
def build_context(results):
    blocks = []
    for i, r in enumerate(results, 1):
        text = r["content"]["text"]
        blocks.append(f"[문서 {i}]\n{text}")
    return "\n\n".join(blocks)


context = build_context(results)
print(context)

## 셀 7 — 프롬프트 증강

"검색된 컨텍스트 + 사용자 질문"을 하나의 프롬프트로 합칩니다.

In [ ]:
def augment_prompt(context, question):
    return (
        "다음 컨텍스트를 참고하여 질문에 답하세요. "
        "컨텍스트에 없는 내용은 모른다고 답하세요.\n\n"
        f"[컨텍스트]\n{context}\n\n"
        f"[질문]\n{question}"
    )


prompt = augment_prompt(context, question)
print(prompt)

## 셀 8 — LLM 답변 생성 (converse)

증강된 프롬프트를 Bedrock `converse` API로 LLM에 전달하고 답변을 출력합니다.

In [ ]:
def generate_answer(prompt):
    resp = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 1000, "temperature": 0.2},
    )
    return resp["output"]["message"]["content"][0]["text"]


answer = generate_answer(prompt)
print(answer)